# 03 · Deep Learning Multimodal — CLIP Congelado + Sparse Autoencoder (SAE)
**Proyecto:** GeoVisionCLIP-Cali  
**Fase:** 3 — Foundation Model (GeoRSCLIP) + SAE + Validación Retrieval + Exportación de Artefactos  

---
### Arquitectura
```
Zarr[i] (4×224×224)  ──►  GeoRSCLIP Image Encoder (CONGELADO)  ──►  embedding_visual (512)
                                                                             │
                                                                    SAE Encoder (512→2048)
                                                                             │
                                                                    latent z (2048) · ReLU
                                                                             │
                                                                    SAE Decoder (2048→512)
                                                                             │
                                                                    x_hat (512) ← MSE + L1
```
**KPIs objetivo:** Reconstrucción ≥ 70% · Sparsity Ratio ≥ 0.70 · Recall@1 ≥ 0.45

## 0 · Dependencias

In [1]:
#!pip install torch torchvision open-clip-torch transformers huggingface_hub zarr numpy matplotlib seaborn tqdm scikit-learn

## 1 · BLOQUE MANDATORIO — Forzado de GPU de Alto Rendimiento (RTX 3050)

In [2]:
import torch

# ─────────────────────────────────────────────────────────────────────────────
# BLOQUE DE OPTIMIZACIÓN GPU — OBLIGATORIO — RTX 3050 Laptop 6 GB VRAM
# ─────────────────────────────────────────────────────────────────────────────
torch.cuda.set_per_process_memory_fraction(0.9)   # Usa hasta el 90% de los ~6.4 GB VRAM
torch.backends.cuda.matmul.allow_tf32 = True       # TF32 en matmul → +40% velocidad
torch.backends.cudnn.allow_tf32       = True       # TF32 en cuDNN
torch.backends.cudnn.benchmark        = True       # Auto-tune del kernel más rápido
torch.backends.cudnn.deterministic    = False      # Permite kernels no deterministas
# ─────────────────────────────────────────────────────────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo activo: {device}")
if device.type == "cuda":
    print(f"Nombre GPU           : {torch.cuda.get_device_name(0)}")
    print(f"VRAM total           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"TF32 matmul          : {torch.backends.cuda.matmul.allow_tf32}")
    print(f"cuDNN benchmark      : {torch.backends.cudnn.benchmark}")
else:
    print("⚠ GPU no disponible. El entrenamiento será lento en CPU.")

Dispositivo activo: cuda
Nombre GPU           : NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM total           : 6.44 GB
TF32 matmul          : True
cuDNN benchmark      : True


In [3]:
import os
import json
import hashlib
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import numpy as np
import zarr
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.cuda.amp as amp

import open_clip
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")

PyTorch : 2.4.1+cu118
CUDA    : 11.8


## 2 · Configuración Global

In [ ]:
# ─── Rutas ────────────────────────────────────────────────────────────────
ZARR_PATH      = Path(r"D:\analitica\procesado_zarr\sentinel2_224.zarr")
DATASET_JSONL  = Path(r"D:\analitica\dataset_multimodal_en.jsonl")
OUTPUT_DIR     = Path(r"D:\analitica\outputs_geovision")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Ruta local del checkpoint GeoRSCLIP (se descarga una vez)
GEORSCLIP_DIR  = Path(r"D:\analitica\GeoRSCLIP\ckpt")          # donde clonas el repo
GEORSCLIP_CKPT = GEORSCLIP_DIR / "RS5M_ViT-B-32.pt"      # checkpoint ViT-B-32


# ─── Hiperparámetros SAE ──────────────────────────────────────────────────
EMBED_DIM       = 512     # dimensión de salida del Image Encoder de CLIP
EXPANSION_DIM   = 2048    # sobreexpansión para poda de neuronas
L1_WEIGHT       = 1e-3    # λ para regularización L1
SPARSITY_THRESH = 0.01    # umbral de inactividad de neurona

# ─── Entrenamiento ────────────────────────────────────────────────────────
BATCH_SIZE      = 64      # ajustado para 6 GB VRAM con GeoRSCLIP frozen
N_EPOCHS        = 50
LR              = 3e-4
RANDOM_SEED     = 42

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Configuración SAE:")
print(f"  Arquitectura : {EMBED_DIM} → {EXPANSION_DIM} → {EMBED_DIM}")
print(f"  λ L1         : {L1_WEIGHT}")
print(f"  Batch size   : {BATCH_SIZE}")
print(f"  Épocas       : {N_EPOCHS}")

Configuración SAE:
  Arquitectura : 512 → 2048 → 512
  λ L1         : 0.001
  Batch size   : 64
  Épocas       : 50


## 3 · Carga del Foundation Model GeoRSCLIP (Congelado)

In [ ]:
# ─── Cargar GeoRSCLIP desde HuggingFace / open_clip ───────────────────────
# Regla de Oro: CLIP viene preentrenado. NO se entrena desde cero.
# Text Encoder SIEMPRE congelado. Image Encoder SIEMPRE en modo eval.

from huggingface_hub import hf_hub_download

# ─── GeoRSCLIP se distribuye como checkpoint .pt, NO como hf-hub open_clip ─
# Descarga automática si no existe localmente

GEORSCLIP_DIR.mkdir(parents=True, exist_ok=True)

if not GEORSCLIP_CKPT.exists():
    print("Descargando RS5M_ViT-B-32.pt desde Zilun/GeoRSCLIP...")
    downloaded = hf_hub_download(
        repo_id   = "Zilun/GeoRSCLIP",
        filename  = "RS5M_ViT-B-32.pt",
        local_dir = str(GEORSCLIP_DIR),
    )
    print(f"  Guardado en: {downloaded}")
else:
    print(f"Checkpoint ya existe: {GEORSCLIP_CKPT}")

# ─── Cargar arquitectura base ViT-B/32 y luego inyectar pesos GeoRSCLIP ───
print("Construyendo modelo ViT-B/32 + cargando pesos GeoRSCLIP...")
clip_model, _, clip_preprocess = open_clip.create_model_and_transforms(
    "ViT-B/32", pretrained="openai"
)
tokenizer = open_clip.get_tokenizer("ViT-B-32")

checkpoint = torch.load(str(GEORSCLIP_CKPT), map_location="cpu")
msg = clip_model.load_state_dict(checkpoint, strict=False)
print(f"  Pesos cargados. Missing: {len(msg.missing_keys)} | Unexpected: {len(msg.unexpected_keys)}")
print("✅ GeoRSCLIP (ViT-B/32) cargado correctamente.")

# ─── CONGELAR todos los parámetros de CLIP ────────────────────────────────
clip_model.eval()
for param in clip_model.parameters():
    param.requires_grad = False

clip_model = clip_model.to(device)

# ─── Fijar EMBED_DIM dinámicamente según el modelo cargado ───────────────
with torch.no_grad():
    _dummy = torch.zeros(1, 3, 224, 224).to(device)
    EMBED_DIM     = clip_model.encode_image(_dummy).shape[-1]
    EXPANSION_DIM = EMBED_DIM * EXPANSION_FACTOR

n_clip_params = sum(p.numel() for p in clip_model.parameters())
print(f"\n  Parámetros CLIP (congelados): {n_clip_params:,}")
print(f"  EMBED_DIM detectado         : {EMBED_DIM}")
print(f"  EXPANSION_DIM               : {EXPANSION_DIM}")
print(f"  Arquitectura SAE            : {EMBED_DIM} → {EXPANSION_DIM} → {EMBED_DIM}")
print(f"  Modo                        : eval() — completamente congelado")

Cargando GeoRSCLIP desde HuggingFace (Zilun/GeoRSCLIP)...
⚠ GeoRSCLIP no disponible (Failed initial config/weights load from HF Hub Zilun/GeoRSCLIP: Failed to download file (open_clip_config.json) for Zilun/GeoRSCLIP. Last error: 404 Client Error. (Request ID: Root=1-6a00eca9-6933f939708bc7834e16c4ca;f004a60f-cd6a-415a-a5e9-b009504afc53)

Entry Not Found for url: https://huggingface.co/Zilun/GeoRSCLIP/resolve/main/open_clip_config.json.).
  Fallback → RemoteCLIP (ViT-L-14 preentrenado en imágenes satelitales)
✅ RemoteCLIP (ViT-L-14) cargado como fallback.

  Parámetros CLIP (congelados): 427,616,513
  Modo               : eval() — Text Encoder + Image Encoder congelados


## 4 · Dataset PyTorch

In [6]:
with open(DATASET_JSONL, "r", encoding="utf-8") as f:
    for i in range(5):
        print(json.loads(next(f)))

{'tile_id': 0, 'zarr_reference': 'D:\\analitica\\procesado_zarr\\sentinel2_224.zarr[0]', 'scene_id': 'S2B_18NUJ_20200127_0_L2A', 'date': '2020-01-27', 'lat_centroid': 3.121542516671604, 'lon_centroid': -76.3055568274979, 'val_no2': 2.4556313426941824e-05, 'val_so2': 2.4556313426941824e-05, 'val_o3': 2.4556313426941824e-05, 'caption_en': 'Satellite image of Cali, Colombia, with low nitrogen dioxide pollution, low sulfur dioxide concentration, and low ozone levels.'}
{'tile_id': 1, 'zarr_reference': 'D:\\analitica\\procesado_zarr\\sentinel2_224.zarr[1]', 'scene_id': 'S2B_18NUJ_20200127_1_L2A', 'date': '2020-01-27', 'lat_centroid': 3.121542516671604, 'lon_centroid': -76.3055568274979, 'val_no2': 2.4556313426941824e-05, 'val_so2': 2.4556313426941824e-05, 'val_o3': 2.4556313426941824e-05, 'caption_en': 'Satellite image of Cali, Colombia, with low nitrogen dioxide pollution, low sulfur dioxide concentration, and low ozone levels.'}
{'tile_id': 2, 'zarr_reference': 'D:\\analitica\\procesado_z

In [7]:
class GeoVisionDataset(Dataset):

    def __init__(
        self,
        jsonl_path: Path,
        zarr_path: Path,
        tokenizer,
        max_text_len: int = 77
    ):

        self.zarr_store   = zarr.open(str(zarr_path), mode="r")
        self.tokenizer    = tokenizer
        self.max_text_len = max_text_len

        self.records = []

        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line.strip())

                tile_id = rec.get("tile_id")

                if tile_id is not None:
                    self.records.append(rec)

        if len(self.records) == 0:
            raise ValueError("Dataset vacío.")

        print(f"Dataset cargado: {len(self.records):,} pares imagen-texto.")

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):

        rec = self.records[idx]

        zarr_idx = int(rec["tile_id"])

        # (4,224,224)
        img_np = self.zarr_store[zarr_idx]

        # RGB para CLIP
        img_rgb = img_np[:3].astype(np.float32) / 10000.0  # Sentinel-2: [0,10000] → [0,1]

        # Normalización ImageNet que espera CLIP
        CLIP_MEAN = np.array([0.48145466, 0.4578275,  0.40821073], dtype=np.float32)
        CLIP_STD  = np.array([0.26862954, 0.26130258, 0.27577711], dtype=np.float32)
        img_rgb = (img_rgb - CLIP_MEAN[:, None, None]) / CLIP_STD[:, None, None]

        img_t = torch.from_numpy(img_rgb).float()  # (3, 224, 224)

        # Texto
        text = rec["caption_en"]

        text_tokens = self.tokenizer([text])[0]

        meta = {
            "scene_id": rec.get("scene_id", ""),
            "date": rec.get("date", ""),
            "lat_centroid": rec.get("lat_centroid", 0.0),
            "lon_centroid": rec.get("lon_centroid", 0.0),
            "val_no2": rec.get("val_no2", 0.0),
            "val_so2": rec.get("val_so2", 0.0),
            "val_o3": rec.get("val_o3", 0.0),
        }

        return img_t, text_tokens, meta

# ── Instanciar dataset y dataloaders ──────────────────────────────────────
full_dataset = GeoVisionDataset(
    jsonl_path=DATASET_JSONL,
    zarr_path=ZARR_PATH,
    tokenizer=tokenizer,
)

n_total = len(full_dataset)
n_val   = max(100, int(0.1 * n_total))
n_train = n_total - n_val

train_ds, val_ds = torch.utils.data.random_split(
    full_dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(RANDOM_SEED)
)

# num_workers=0 es obligatorio en Windows + Zarr + Jupyter
# prefetch_factor y persistent_workers NO se usan con workers=0
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

print(f"Train: {n_train:,} | Val: {n_val:,}")
print(f"Batches por época (train): {len(train_loader)}")

Dataset cargado: 363 pares imagen-texto.
Train: 263 | Val: 100
Batches por época (train): 5


## 5 · Arquitectura del Sparse Autoencoder (SAE)

In [8]:
class SparseAutoencoder(nn.Module):
    """
    SAE sobreexpansivo: 512 → 2048 → 512.
    La sobreexpansión permite descubrir y podar características latentes
    no relevantes (ej. nubes, sombras de altitud, ruido de sensor).
    """

    def __init__(self, input_dim: int = 512, expansion_dim: int = 2048):
        super().__init__()
        # Encoder: proyección sobreexpansiva
        self.encoder = nn.Linear(input_dim, expansion_dim, bias=True)
        self.relu    = nn.ReLU()
        # Decoder: proyección de vuelta al espacio original
        self.decoder = nn.Linear(expansion_dim, input_dim, bias=True)

        # Inicialización Kaiming para estabilidad con ReLU
        nn.init.kaiming_normal_(self.encoder.weight, nonlinearity="relu")
        nn.init.kaiming_normal_(self.decoder.weight, nonlinearity="relu")
        nn.init.zeros_(self.encoder.bias)
        nn.init.zeros_(self.decoder.bias)

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        x          : (B, 512) embeddings del Image Encoder CLIP
        latents    : (B, 2048) representación dispersa
        reconstructed : (B, 512) reconstrucción
        """
        latents      = self.relu(self.encoder(x))
        reconstructed = self.decoder(latents)
        return reconstructed, latents


def sae_loss(x: torch.Tensor,
             reconstructed: torch.Tensor,
             latents: torch.Tensor,
             l1_weight: float = 1e-3) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Loss total = MSE (reconstrucción) + λ·L1 (sparsity).
    Devuelve (loss_total, mse, l1) para trazabilidad por curva.
    """
    mse   = nn.MSELoss()(reconstructed, x)
    l1    = torch.mean(torch.abs(latents))
    total = mse + l1_weight * l1
    return total, mse, l1


def sparsity_ratio(latents: torch.Tensor,
                   threshold: float = SPARSITY_THRESH) -> float:
    """
    Fracción de neuronas inactivas (z < threshold).
    KPI objetivo: ≥ 0.70
    """
    return float((latents < threshold).float().mean().item())


def reconstruction_score(x: torch.Tensor,
                          reconstructed: torch.Tensor) -> float:
    """
    Porcentaje de varianza explicada (1 - MSE / Var(x)).
    KPI objetivo: ≥ 0.70
    """
    mse     = nn.MSELoss()(reconstructed, x).item()
    var_x   = x.var().item()
    if var_x == 0:
        return 0.0
    return max(0.0, 1.0 - mse / var_x)


# Instanciar SAE
sae = SparseAutoencoder(input_dim=EMBED_DIM, expansion_dim=EXPANSION_DIM).to(device)
optimizer = torch.optim.Adam(sae.parameters(), lr=LR, weight_decay=0.0)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS, eta_min=1e-5
)

n_sae_params = sum(p.numel() for p in sae.parameters() if p.requires_grad)
print(f"SAE instanciado. Parámetros entrenables: {n_sae_params:,}")
print(f"Arquitectura: {EMBED_DIM} → {EXPANSION_DIM} → {EMBED_DIM}")

SAE instanciado. Parámetros entrenables: 2,099,712
Arquitectura: 512 → 2048 → 512


## 6 · Función de Extracción de Embeddings CLIP (Image Encoder Congelado)

In [9]:
@torch.no_grad()
def extract_image_embeddings(images: torch.Tensor,
                              clip_model: nn.Module) -> torch.Tensor:
    """
    Pasa imágenes (B, 3, 224, 224) por el Image Encoder congelado.
    Devuelve embeddings L2-normalizados de forma (B, 512).
    """
    images = images.to(device)
    feats  = clip_model.encode_image(images)           # (B, 512)
    feats  = feats / feats.norm(dim=-1, keepdim=True)  # L2 normalize
    return feats.float()


@torch.no_grad()
def extract_text_embeddings(tokens: torch.Tensor,
                             clip_model: nn.Module) -> torch.Tensor:
    """
    Pasa tokens de texto por el Text Encoder congelado.
    Devuelve embeddings L2-normalizados de forma (B, 512).
    """
    tokens = tokens.to(device)
    feats  = clip_model.encode_text(tokens)            # (B, 512)
    feats  = feats / feats.norm(dim=-1, keepdim=True)
    return feats.float()


print("Funciones de extracción CLIP definidas.")

Funciones de extracción CLIP definidas.


## 7 · Bucle de Entrenamiento del SAE

In [10]:
# Forzar modo alto rendimiento en GPU
torch.cuda.set_per_process_memory_fraction(0.9)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

# Verificar que CUDA está activo correctamente
print(f"TF32 matmul: {torch.backends.cuda.matmul.allow_tf32}")
print(f"cuDNN benchmark: {torch.backends.cudnn.benchmark}")

TF32 matmul: True
cuDNN benchmark: True


In [11]:
import time

# ── Print de diagnóstico inicial ──────────────────────────────────────────
print("Entrando al entrenamiento...")
print(f"  device       : {device}")
print(f"  train batches: {len(train_loader)}")
print(f"  val batches  : {len(val_loader)}")

# ── Historial para curvas de pérdida ──────────────────────────────────────
history = {
    "epoch"       : [],
    "loss_total"  : [],
    "loss_mse"    : [],
    "loss_l1"     : [],
    "sparsity"    : [],
    "recon_score" : [],
    "val_loss"    : [],
    "val_sparsity": [],
    "val_recon"   : [],
}

scaler = amp.GradScaler()  # Precisión mixta para RTX 3050

best_val_loss = float("inf")
CHECKPOINT_PATH = OUTPUT_DIR / "sae_cali_best.pt"
FINAL_PATH      = OUTPUT_DIR / "sae_cali_final.pt"


for epoch in range(1, N_EPOCHS + 1):
    # ── TRAIN ────────────────────────────────────────────────────────────
    sae.train()
    epoch_loss = epoch_mse = epoch_l1 = epoch_sp = epoch_rs = 0.0
    n_batches  = 0

    for batch_idx, (imgs, tokens, _) in enumerate(tqdm(
        train_loader,
        desc=f"Época {epoch}/{N_EPOCHS} [TRAIN]",
        leave=False
    )):
        # ── Diagnóstico solo en el primer batch de la primera época ──────
        if epoch == 1 and batch_idx == 0:
            print(f"\n[Diagnóstico batch 0]")
            print(f"  imgs.shape : {imgs.shape}  dtype={imgs.dtype}")
            print(f"  imgs range : [{imgs.min():.3f}, {imgs.max():.3f}]")

        # 1. Mover a device explícitamente
        imgs   = imgs.to(device, non_blocking=False)
        tokens = tokens.to(device, non_blocking=False)

        # 2. Extraer embeddings con Image Encoder congelado
        t0 = time.time()
        with torch.no_grad():
            emb = extract_image_embeddings(imgs, clip_model)  # (B, 512)

        if epoch == 1 and batch_idx == 0:
            print(f"  emb.shape  : {emb.shape}  tiempo={time.time()-t0:.2f}s")

        # 3. Forward SAE con precisión mixta
        optimizer.zero_grad()
        with amp.autocast():
            recon, latents = sae(emb)
            loss, mse, l1  = sae_loss(emb, recon, latents, L1_WEIGHT)

        # 4. Backward + optimización
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(sae.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        # 5. Métricas de batch
        with torch.no_grad():
            sp = sparsity_ratio(latents.detach())
            rs = reconstruction_score(emb.detach(), recon.detach())

        epoch_loss += loss.item()
        epoch_mse  += mse.item()
        epoch_l1   += l1.item()
        epoch_sp   += sp
        epoch_rs   += rs
        n_batches  += 1

    scheduler.step()

    # ── VALIDACIÓN ───────────────────────────────────────────────────────
    sae.eval()
    val_loss_acc = val_sp_acc = val_rs_acc = 0.0
    n_val_batches = 0

    with torch.no_grad():
        for imgs, tokens, _ in val_loader:
            imgs  = imgs.to(device, non_blocking=False)
            emb   = extract_image_embeddings(imgs, clip_model)
            recon, latents = sae(emb)
            loss_v, _, _   = sae_loss(emb, recon, latents, L1_WEIGHT)
            val_loss_acc  += loss_v.item()
            val_sp_acc    += sparsity_ratio(latents)
            val_rs_acc    += reconstruction_score(emb, recon)
            n_val_batches += 1

    # ── Promediar y registrar ─────────────────────────────────────────────
    avg    = lambda x, n: x / n if n > 0 else 0.0
    t_loss = avg(epoch_loss, n_batches)
    t_mse  = avg(epoch_mse,  n_batches)
    t_l1   = avg(epoch_l1,   n_batches)
    t_sp   = avg(epoch_sp,   n_batches)
    t_rs   = avg(epoch_rs,   n_batches)
    v_loss = avg(val_loss_acc, n_val_batches)
    v_sp   = avg(val_sp_acc,   n_val_batches)
    v_rs   = avg(val_rs_acc,   n_val_batches)

    history["epoch"].append(epoch)
    history["loss_total"].append(t_loss)
    history["loss_mse"].append(t_mse)
    history["loss_l1"].append(t_l1)
    history["sparsity"].append(t_sp)
    history["recon_score"].append(t_rs)
    history["val_loss"].append(v_loss)
    history["val_sparsity"].append(v_sp)
    history["val_recon"].append(v_rs)

    # ── Guardar mejor checkpoint ──────────────────────────────────────────
    if v_loss < best_val_loss:
        best_val_loss = v_loss
        torch.save({
            "epoch"       : epoch,
            "model_state" : sae.state_dict(),
            "optim_state" : optimizer.state_dict(),
            "val_loss"    : v_loss,
            "config"      : {"input_dim": EMBED_DIM, "expansion_dim": EXPANSION_DIM},
        }, CHECKPOINT_PATH)

    # ── Log cada 5 épocas ─────────────────────────────────────────────────
    if epoch % 5 == 0 or epoch == 1:
        kpi_recon = "✅" if t_rs >= 0.70 else "⏳"
        kpi_sp    = "✅" if t_sp >= 0.70 else "⏳"
        print(
            f"[Ep {epoch:3d}] "
            f"Loss={t_loss:.4f} MSE={t_mse:.4f} L1={t_l1:.4f} | "
            f"Recon={t_rs:.2%} {kpi_recon} | "
            f"Sparsity={t_sp:.2%} {kpi_sp} | "
            f"Val={v_loss:.4f}"
        )

# Guardar checkpoint final
torch.save(sae.state_dict(), FINAL_PATH)
print(f"\n✅ Entrenamiento completado.")
print(f"   Mejor checkpoint : {CHECKPOINT_PATH}")
print(f"   Checkpoint final  : {FINAL_PATH}")

Entrando al entrenamiento...
  device       : cuda
  train batches: 5
  val batches  : 2


Época 1/50 [TRAIN]:   0%|          | 0/5 [00:00<?, ?it/s]

KeyError: 97

## 8 · Trazabilidad Visual — Curvas de Entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_x = history["epoch"]

# ── 1. Loss total: Train vs Val ────────────────────────────────────────────
ax = axes[0]
ax.plot(epochs_x, history["loss_total"], label="Train Loss Total", color="#3a86ff")
ax.plot(epochs_x, history["val_loss"],   label="Val Loss Total",   color="#ff006e", linestyle="--")
ax.set_title("Loss Total (MSE + λ·L1)", fontsize=11)
ax.set_xlabel("Época")
ax.set_ylabel("Loss")
ax.legend()
sns.despine(ax=ax)

# ── 2. MSE vs L1 (Train) ──────────────────────────────────────────────────
ax = axes[1]
ax.plot(epochs_x, history["loss_mse"], label="MSE Reconstrucción", color="#8338ec")
ax.plot(epochs_x, history["loss_l1"],  label="L1 Regularización",  color="#fb5607")
ax.set_title("Descomposición del Loss", fontsize=11)
ax.set_xlabel("Época")
ax.set_ylabel("Loss")
ax.legend()
sns.despine(ax=ax)

# ── 3. KPIs: Reconstrucción y Sparsity ────────────────────────────────────
ax = axes[2]
ax.plot(epochs_x, [s * 100 for s in history["recon_score"]],
        label="Recon Score (Train) %", color="#06d6a0")
ax.plot(epochs_x, [s * 100 for s in history["val_recon"]],
        label="Recon Score (Val) %", color="#06d6a0", linestyle="--")
ax.plot(epochs_x, [s * 100 for s in history["sparsity"]],
        label="Sparsity Ratio (Train) %", color="#ffd166")
ax.axhline(70, color="gray", linestyle=":", linewidth=1.2, label="KPI 70%")
ax.set_title("KPIs: Reconstrucción y Sparsity", fontsize=11)
ax.set_xlabel("Época")
ax.set_ylabel("%")
ax.set_ylim(0, 110)
ax.legend(fontsize=8)
sns.despine(ax=ax)

fig.suptitle("Curvas de Entrenamiento — Sparse Autoencoder (SAE) GeoVisionCLIP-Cali",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("curvas_entrenamiento_sae.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico guardado: curvas_entrenamiento_sae.png")

# ── Reporte de KPIs finales ────────────────────────────────────────────────
final_recon = history["recon_score"][-1] if history["recon_score"] else 0
final_sp    = history["sparsity"][-1]    if history["sparsity"] else 0
print(f"\nKPIs finales (última época):")
print(f"  Reconstrucción : {final_recon:.2%}  {'✅ ≥70%' if final_recon >= 0.70 else '❌ <70%'}")
print(f"  Sparsity Ratio : {final_sp:.2%}     {'✅ ≥70%' if final_sp >= 0.70 else '❌ <70%'}")

## 9 · Trazabilidad Visual — Recortes RGB con Texto Autogenerado

In [ ]:
CLIP_MEAN_VIS = np.array([0.48145466, 0.4578275,  0.40821073])
CLIP_STD_VIS  = np.array([0.26862954, 0.26130258, 0.27577711])

def denorm_rgb(tensor_3c: np.ndarray) -> np.ndarray:
    """Desnormaliza (3, 224, 224) → (224, 224, 3) uint8."""
    t = tensor_3c.copy()
    for i in range(3):
        t[i] = t[i] * CLIP_STD_VIS[i] + CLIP_MEAN_VIS[i]
    t = np.clip(t, 0, 1)
    return (np.transpose(t, (1, 2, 0)) * 255).astype(np.uint8)


# Seleccionar 5 muestras del conjunto de validación
sample_indices = np.random.choice(len(val_ds), size=min(5, len(val_ds)), replace=False)
store_vis = zarr.open(str(ZARR_PATH), mode="r")

# Leer dataset JSONL para obtener textos
records_all: List[Dict] = []
with open(DATASET_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        records_all.append(json.loads(line.strip()))

fig, axes = plt.subplots(1, len(sample_indices), figsize=(5 * len(sample_indices), 5))
if len(sample_indices) == 1:
    axes = [axes]

for ax, idx in zip(axes, sample_indices):
    rec       = records_all[val_ds.indices[idx]]
    zarr_idx  = int(rec["id_tile"])
    tensor_np = store_vis[zarr_idx]      # (4, 224, 224)
    img_rgb   = denorm_rgb(tensor_np[:3])
    texto     = rec["texto_espanol"]

    ax.imshow(img_rgb)
    ax.set_title(f"id={zarr_idx} | {rec.get('fecha','')}", fontsize=7, pad=4)
    ax.set_xlabel("\n".join([texto[i:i+45] for i in range(0, min(len(texto), 135), 45)]),
                  fontsize=6.5)
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

fig.suptitle("Muestras de Validación — RGB Sentinel-2 + Texto Autogenerado (español)",
             fontsize=11, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("muestras_validacion_textos.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico guardado: muestras_validacion_textos.png")

## 10 · Validación de Retrieval — Recall@1 y Recall@5

In [ ]:
@torch.no_grad()
def compute_recall_at_k(sae_model: nn.Module,
                         clip_model: nn.Module,
                         loader: DataLoader,
                         k_values: List[int] = [1, 5]) -> Dict[int, float]:
    """
    Calcula Recall@K comparando similitud coseno entre embeddings
    imagen (procesados por SAE) y texto (Text Encoder CLIP congelado).
    """
    sae_model.eval()

    all_img_emb  = []
    all_txt_emb  = []

    for imgs, tokens, _ in tqdm(loader, desc="Extrayendo embeddings para Recall@K"):
        img_emb = extract_image_embeddings(imgs, clip_model)       # (B, 512)
        txt_emb = extract_text_embeddings(tokens, clip_model)      # (B, 512)

        # Pasar embeddings imagen por el SAE (extracción de latentes)
        _, img_latents = sae_model(img_emb)                         # (B, 2048)
        # Para retrieval, usamos los embeddings reconstruidos (vuelta al espacio 512)
        img_recon, _ = sae_model(img_emb)                           # (B, 512)
        img_recon = img_recon / img_recon.norm(dim=-1, keepdim=True)

        all_img_emb.append(img_recon.cpu())
        all_txt_emb.append(txt_emb.cpu())

    img_matrix = torch.cat(all_img_emb, dim=0)   # (N, 512)
    txt_matrix = torch.cat(all_txt_emb, dim=0)   # (N, 512)

    # Matriz de similitud coseno (N×N)
    sim_matrix = img_matrix @ txt_matrix.T         # (N, N)

    recalls = {}
    N = sim_matrix.shape[0]
    ground_truth = torch.arange(N)  # el par correcto es la diagonal

    for k in k_values:
        topk_indices = sim_matrix.topk(k, dim=1).indices   # (N, k)
        correct = topk_indices.eq(ground_truth.unsqueeze(1)).any(dim=1)
        recalls[k] = correct.float().mean().item()

    return recalls


recalls = compute_recall_at_k(sae, clip_model, val_loader, k_values=[1, 5])

print("\n── Métricas de Retrieval (Validación) ──────────────────")
print(f"  Recall@1 : {recalls[1]:.4f}  {'✅ ≥0.45' if recalls[1] >= 0.45 else '❌ <0.45 (revisar entrenamiento)'}")
print(f"  Recall@5 : {recalls[5]:.4f}")

## 11 · Exportación para Martín (AFE/AFC) — `embeddings_visuales_sae.pt` + JSON Interpretabilidad

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# EXPORTACIÓN PARA MARTÍN — Análisis Factorial Exploratorio/Confirmatorio
# Recibe: matriz de embeddings SAE (n × 512) + mapa de activaciones latentes
# ═══════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def export_for_martin(sae_model: nn.Module,
                       clip_model: nn.Module,
                       full_loader: DataLoader,
                       output_dir: Path) -> None:
    """
    Genera:
    1. embeddings_visuales_sae.pt  : tensor (n × 512) de embeddings reconstruidos por el SAE.
    2. interpretabilidad_latentes.json : mapa de qué índices del espacio latente (2048)
       se activan más por cada clase/contaminante.
    """
    sae_model.eval()

    all_reconstructed = []
    all_latents       = []
    all_meta          = []

    for imgs, _, meta in tqdm(full_loader, desc="Extrayendo para Martín"):
        img_emb   = extract_image_embeddings(imgs, clip_model)      # (B, 512)
        recon, z  = sae_model(img_emb)                               # (B,512), (B,2048)
        all_reconstructed.append(recon.cpu())
        all_latents.append(z.cpu())
        all_meta.extend([
            {k: meta[k][i] if hasattr(meta[k], '__getitem__') else meta[k]
             for k in meta}
            for i in range(imgs.shape[0])
        ])

    emb_matrix    = torch.cat(all_reconstructed, dim=0)   # (n, 512)
    latent_matrix = torch.cat(all_latents, dim=0)          # (n, 2048)

    # 1. Guardar embeddings para AFE/AFC
    emb_path = output_dir / "embeddings_visuales_sae.pt"
    torch.save(emb_matrix, emb_path)
    print(f"✅ embeddings_visuales_sae.pt guardado: shape {tuple(emb_matrix.shape)}")

    # 2. JSON de interpretabilidad: top-20 neuronas latentes por clase
    # Clasificar según umbral percentil de NO2 en los metadatos
    CLASES = ["no2_bajo", "no2_moderado", "no2_alto", "no2_critico",
              "so2_activo", "ozono_anomalo"]

    def classify_sample(m: Dict) -> str:
        no2 = float(m.get("val_no2") or 0)
        so2 = float(m.get("val_so2") or 0)
        if no2 > 1e-4:
            return "no2_critico"
        elif no2 > 5e-5:
            return "no2_alto"
        elif no2 > 2e-5:
            return "no2_moderado"
        elif so2 > 5e-6:
            return "so2_activo"
        else:
            return "no2_bajo"

    class_labels = [classify_sample(m) for m in all_meta]
    interp_map   = {}

    for clase in CLASES:
        indices = [i for i, l in enumerate(class_labels) if l == clase]
        if len(indices) == 0:
            interp_map[clase] = []
            continue
        class_latents = latent_matrix[indices]       # (n_clase, 2048)
        mean_activation = class_latents.mean(dim=0)  # (2048,)
        top20_indices = mean_activation.topk(20).indices.tolist()
        interp_map[clase] = {
            "top_20_indices_latentes": top20_indices,
            "n_muestras"             : len(indices),
            "activacion_media_max"   : float(mean_activation.max()),
        }

    interp_path = output_dir / "interpretabilidad_latentes.json"
    with open(interp_path, "w") as f:
        json.dump(interp_map, f, indent=2)
    print(f"✅ interpretabilidad_latentes.json guardado: {interp_path}")

    return emb_matrix


# DataLoader completo (train + val) para exportación
full_loader_export = DataLoader(
    full_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True,
)

emb_for_martin = export_for_martin(sae, clip_model, full_loader_export, OUTPUT_DIR)

## 12 · Exportación para Luz Ángela (GRU Temporal) — Tensor de Ventanas de 8 Fechas

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# EXPORTACIÓN PARA LUZ ÁNGELA — Secuencias Temporales para nn.GRU
# Formato: (n_ventanas, 8, 2048)  → directamente consumible por GRU
#   n_ventanas : número de secuencias de 8 fechas extraídas
#   8          : pasos temporales (ventana deslizante)
#   2048       : dimensión del espacio latente del SAE
# ═══════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def export_for_luz_angela(sae_model: nn.Module,
                           clip_model: nn.Module,
                           records: List[Dict],
                           zarr_path: Path,
                           output_dir: Path,
                           window_size: int = 8,
                           stride: int = 1) -> None:
    """
    Reorganiza los embeddings latentes del SAE en ventanas temporales
    de `window_size` fechas consecutivas por escena espacial.
    Output: secuencias_gru.pt  → tensor (n_ventanas, window_size, 2048)
    """
    sae_model.eval()
    store = zarr.open(str(zarr_path), mode="r")

    # Agrupar registros por centroide espacial (redondeado a 0.05°)
    from collections import defaultdict
    spatial_groups: Dict[Tuple, List[Dict]] = defaultdict(list)

    for rec in records:
        lat_r = round(float(rec.get("lat_centroide", 0)), 2)
        lon_r = round(float(rec.get("lon_centroide", 0)), 2)
        spatial_groups[(lat_r, lon_r)].append(rec)

    # Ordenar cada grupo por fecha
    for key in spatial_groups:
        spatial_groups[key].sort(key=lambda r: r.get("fecha", ""))

    all_windows = []
    window_meta = []

    for (lat_r, lon_r), tile_series in tqdm(
        spatial_groups.items(), desc="Generando ventanas temporales"
    ):
        if len(tile_series) < window_size:
            continue

        # Extraer latentes para toda la serie de esta localización
        series_latents = []
        for rec in tile_series:
            zarr_idx = int(rec["id_tile"])
            img_np   = store[zarr_idx][:3]                    # (3, 224, 224)
            img_t    = torch.from_numpy(img_np).float().unsqueeze(0).to(device)
            emb      = extract_image_embeddings(img_t, clip_model)   # (1, 512)
            _, z     = sae_model(emb)                                  # (1, 2048)
            series_latents.append(z.cpu().squeeze(0))                  # (2048,)

        series_tensor = torch.stack(series_latents)  # (T, 2048)
        T = series_tensor.shape[0]

        # Deslizar ventana de window_size con stride
        for start in range(0, T - window_size + 1, stride):
            window = series_tensor[start : start + window_size]  # (8, 2048)
            all_windows.append(window)
            window_meta.append({
                "lat"         : lat_r,
                "lon"         : lon_r,
                "fecha_inicio": tile_series[start].get("fecha", ""),
                "fecha_fin"   : tile_series[start + window_size - 1].get("fecha", ""),
                "window_idx"  : len(all_windows) - 1,
            })

    if not all_windows:
        print("⚠ Sin series temporales suficientes (< 8 fechas por localización).")
        return

    # Tensor final: (n_ventanas, 8, 2048) → listo para nn.GRU
    gru_tensor = torch.stack(all_windows)   # (n_ventanas, window_size, 2048)

    gru_path = output_dir / "secuencias_gru.pt"
    torch.save({
        "tensor"          : gru_tensor,
        "meta"            : window_meta,
        "window_size"     : window_size,
        "latent_dim"      : EXPANSION_DIM,
        "formato_gru"     : "(n_ventanas, seq_len, input_size) → nn.GRU(input_size=2048)",
    }, gru_path)

    print(f"✅ secuencias_gru.pt guardado: shape {tuple(gru_tensor.shape)}")
    print(f"   n_ventanas  : {gru_tensor.shape[0]}")
    print(f"   seq_len     : {gru_tensor.shape[1]} (fechas consecutivas)")
    print(f"   input_size  : {gru_tensor.shape[2]} (dim. latente SAE)")
    print(f"   Uso en GRU  : gru = nn.GRU(input_size=2048, hidden_size=H, batch_first=True)")
    print(f"                 output, _ = gru(tensor)  # tensor shape: {tuple(gru_tensor.shape)}")


# Recargar todos los records para la exportación
records_export: List[Dict] = []
with open(DATASET_JSONL, "r", encoding="utf-8") as f:
    for line in f:
        records_export.append(json.loads(line.strip()))

export_for_luz_angela(
    sae_model  = sae,
    clip_model = clip_model,
    records    = records_export,
    zarr_path  = ZARR_PATH,
    output_dir = OUTPUT_DIR,
    window_size= 8,
    stride     = 1,
)

## 13 · Trazabilidad General — Hash MD5 del Checkpoint Final

In [ ]:
import hashlib

def md5_file(path: Path, chunk_size: int = 1 << 20) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


# ── Calcular MD5 de todos los artefactos clave ────────────────────────────
artifacts = {
    "sae_cali_final.pt"            : FINAL_PATH,
    "sae_cali_best.pt"             : CHECKPOINT_PATH,
    "embeddings_visuales_sae.pt"   : OUTPUT_DIR / "embeddings_visuales_sae.pt",
    "secuencias_gru.pt"            : OUTPUT_DIR / "secuencias_gru.pt",
    "interpretabilidad_latentes.json": OUTPUT_DIR / "interpretabilidad_latentes.json",
}

manifest_update = {}
print("\n── Trazabilidad: MD5 de Artefactos ────────────────────────")
for name, path in artifacts.items():
    if Path(path).exists():
        md5 = md5_file(Path(path))
        manifest_update[name] = {"path": str(path), "md5": md5}
        print(f"  {name:<42} MD5: {md5}")
    else:
        print(f"  ⚠ {name} — archivo no encontrado en {path}")

# Guardar manifest actualizado
manifest_path = OUTPUT_DIR / "manifest_artefactos.json"
with open(manifest_path, "w") as f:
    json.dump({
        "proyecto"    : "GeoVisionCLIP-Cali",
        "arquitectura": f"GeoRSCLIP + SAE ({EMBED_DIM}→{EXPANSION_DIM}→{EMBED_DIM})",
        "kpi_final"   : {
            "recall_at_1"     : recalls.get(1, "N/A"),
            "recall_at_5"     : recalls.get(5, "N/A"),
            "recon_score"     : history["recon_score"][-1] if history["recon_score"] else None,
            "sparsity_ratio"  : history["sparsity"][-1]    if history["sparsity"] else None,
        },
        "artefactos"  : manifest_update,
    }, f, indent=2)

print(f"\n✅ Manifest actualizado: {manifest_path}")
print("\n── Resumen de entregables ──────────────────────────────────")
print("  Para Martín (AFE/AFC):")
print(f"    → embeddings_visuales_sae.pt   (n × 512)")
print(f"    → interpretabilidad_latentes.json")
print("  Para Luz Ángela (GRU Temporal):")
print(f"    → secuencias_gru.pt            (n_ventanas × 8 × 2048)")
print("  Trazabilidad:")
print(f"    → manifest_artefactos.json (MD5 de todos los checkpoints)")
print("\n→ Fase 3 completada. Pipeline GeoVisionCLIP-Cali finalizado.")